# Laboratorio completo de **Data Profiling** y **documentación de datasets**
## Con **pandas** y **ydata-profiling**


Aquí vas a aprender a:

- entender qué es el **data profiling**
- revisar la calidad de un dataset con **pandas**
- generar un reporte automático con **ydata-profiling**
- documentar un dataset de manera profesional
- comparar un dataset **crudo** vs un dataset **mejorado**
- exportar entregables en HTML, CSV, Excel y Markdown

> **Data profiling = conocer, medir y evaluar la calidad del dataset antes de analizarlo o usarlo en modelos.**

## Cómo usar este notebook en Google Colab

1. Abre un notebook nuevo en Colab.
2. Sube este archivo `.ipynb` o ábrelo desde Drive.
3. Ejecuta la celda de instalación.
4. Decide si quieres guardar los resultados:
   - solo en Colab (`/content/...`)
   - o en Google Drive
5. Ejecuta el notebook de arriba hacia abajo.

# 0. Instalación en Google Colab

In [0]:

# ============================================================
# 0. Instalación en Google Colab
# ============================================================

import sys

# Instalación recomendada para notebook / Colab
!{sys.executable} -m pip install -U ydata-profiling[notebook] openpyxl xlsxwriter pyarrow

print("Si Colab te lo pide, reinicia el entorno y vuelve a ejecutar desde arriba.")

 # 1. Imports y configuración

In [0]:

# ============================================================
# 1. Imports y configuración
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ydata_profiling import ProfileReport


## Guardado local o en Google Drive

- guardar los artefactos solo en la sesión de Colab (`/content`)


In [0]:
from google.colab import drive
drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/02_Profiling_Datasets_Pandas")


DATA_DIR = BASE_DIR / "data"
REPORTS_DIR = BASE_DIR / "reports"
DOCS_DIR = BASE_DIR / "docs"

for d in [BASE_DIR, DATA_DIR, REPORTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA_DIR / "ventas_clientes_raw.csv"
CLEAN_CSV = DATA_DIR / "ventas_clientes_clean.csv"
PROFILE_HTML_RAW = REPORTS_DIR / "profile_raw.html"
PROFILE_HTML_CLEAN = REPORTS_DIR / "profile_clean.html"
DATA_DICTIONARY_CSV = DOCS_DIR / "diccionario_datos.csv"
DATA_QUALITY_CSV = DOCS_DIR / "reglas_calidad.csv"
DATASET_SUMMARY_MD = DOCS_DIR / "resumen_dataset.md"
COMPARISON_CSV = DOCS_DIR / "comparacion_raw_vs_clean.csv"

print("Base:", BASE_DIR)
print("Dataset crudo:", RAW_CSV)

# Parte 1. Construcción del dataset de ejemplo

Vamos a crear un dataset sintético de ventas y clientes con problemas deliberados:
- valores nulos
- duplicados
- categorías inconsistentes
- fechas futuras
- edades imposibles
- valores negativos
- emails mal formados

# 3. Crear dataset sintético con problemas de calidad

In [0]:

# ============================================================
# 3. Crear dataset sintético con problemas de calidad
# ============================================================

rng = np.random.default_rng(42)
n = 50000

ciudades = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Cartagena"]
canales = ["web", "app", "tienda", "call_center"]
estados = ["completada", "pendiente", "cancelada", "devuelta"]
segmentos = ["hogar", "pyme", "corporativo"]
productos = ["tarjeta", "seguro", "credito", "ahorro", "inversion"]

df = pd.DataFrame({
    "id_venta": np.arange(1, n + 1),
    "fecha_venta": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, size=n), unit="D"),
    "ciudad": rng.choice(ciudades, size=n, p=[0.28, 0.22, 0.18, 0.12, 0.10, 0.10]),
    "canal": rng.choice(canales, size=n, p=[0.38, 0.32, 0.20, 0.10]),
    "estado": rng.choice(estados, size=n, p=[0.72, 0.12, 0.10, 0.06]),
    "segmento": rng.choice(segmentos, size=n, p=[0.58, 0.28, 0.14]),
    "producto": rng.choice(productos, size=n),
    "edad_cliente": rng.normal(39, 12, size=n).round().astype("int64"),
    "ingreso_mensual": rng.normal(3200, 1200, size=n).round(2),
    "cantidad": rng.integers(1, 8, size=n),
    "precio_unitario": rng.normal(180, 70, size=n).round(2),
    "descuento_pct": np.clip(rng.normal(0.12, 0.08, size=n), 0, 0.60).round(3), # Genero números aleatorios con distribució normal media = 0.12 y desviación estándar= 0.08 y los ajusto para que esten en el rango 0 a 0.60
    "email": [f"cliente{i}@correo.com" for i in range(1, n + 1)]
})

df["monto_bruto"] = (df["cantidad"] * df["precio_unitario"]).round(2)
df["monto_neto"] = (df["monto_bruto"] * (1 - df["descuento_pct"])).round(2)

# Nulos
null_idx = rng.choice(df.index, size=2500, replace=False) #se seleccionan 2500 índices del dataframe de forma aleatoria
df.loc[null_idx[:900], "ciudad"] = None
df.loc[null_idx[900:1500], "ingreso_mensual"] = np.nan
df.loc[null_idx[1500:1900], "edad_cliente"] = np.nan
df.loc[null_idx[1900:2200], "email"] = None
df.loc[null_idx[2200:], "canal"] = None

# Texto inconsistente
idx_text = rng.choice(df.index, size=1200, replace=False)
df.loc[idx_text[:300], "ciudad"] = " bogotá "
df.loc[idx_text[300:600], "ciudad"] = "MEDELLIN"
df.loc[idx_text[600:900], "canal"] = "App"
df.loc[idx_text[900:], "estado"] = "Completada"

# Edades imposibles
idx_edad = rng.choice(df.index, size=150, replace=False)
df.loc[idx_edad[:50], "edad_cliente"] = -5
df.loc[idx_edad[50:100], "edad_cliente"] = 145
df.loc[idx_edad[100:], "edad_cliente"] = 0

# Valores monetarios inválidos
idx_money = rng.choice(df.index, size=180, replace=False)
df.loc[idx_money[:60], "precio_unitario"] = -10
df.loc[idx_money[60:120], "ingreso_mensual"] = -500
df.loc[idx_money[120:], "descuento_pct"] = 1.25

# Fechas futuras
idx_future = rng.choice(df.index, size=200, replace=False)
df.loc[idx_future, "fecha_venta"] = pd.to_datetime("2027-01-01")

# Duplicados
# Duplicar registros con id's diferentes
duplicados = df.sample(300, random_state=7).copy()
duplicados["id_venta"] = df["id_venta"].max() + np.arange(1, len(duplicados) + 1)
df = pd.concat([df, duplicados], ignore_index=True)

#Duplicados exactos
duplicados_negocio = df.sample(200, random_state=11).copy()
df = pd.concat([df, duplicados_negocio], ignore_index=True)

# Emails mal formados
idx_email = rng.choice(df.index, size=250, replace=False)
df.loc[idx_email[:80], "email"] = "sin-arroba.com"
df.loc[idx_email[80:160], "email"] = "cliente@@correo.com"
df.loc[idx_email[160:], "email"] = "cliente sin correo"

df["monto_bruto"] = (df["cantidad"] * df["precio_unitario"]).round(2)
df["monto_neto"] = (df["monto_bruto"] * (1 - df["descuento_pct"])).round(2)

df.to_csv(RAW_CSV, index=False)

print("Filas finales:", len(df))
print("Columnas:", df.shape[1])
print("Archivo guardado en:", RAW_CSV)
df.head()

# Parte 2. Inspección inicial con pandas

Antes del profiling automático, siempre conviene revisar:
- forma del dataset
- tipos de datos
- estadísticas descriptivas
- categorías visibles
- valores raros a primera vista

In [0]:

# ============================================================
# 4. Cargar dataset crudo
# ============================================================

raw = pd.read_csv(RAW_CSV, parse_dates=["fecha_venta"])
print("Shape:", raw.shape)
raw.head()

In [0]:

raw.info()

In [0]:

raw.describe(include="number").T

In [0]:

raw.describe(include="object").T

# Parte 3. Profiling manual con pandas

Aquí construimos un kit simple, útil y entendible

In [0]:
# ============================================================
# 5. Funciones auxiliares
# ============================================================

def percent_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el porcentaje de valores nulos por columna.

    Qué hace:
    - Revisa cada columna del DataFrame.
    - Cuenta qué proporción de sus valores está vacía.
    - Convierte esa proporción a porcentaje.
    - Ordena el resultado de mayor a menor.

    Para qué sirve:
    - Identificar rápidamente qué columnas tienen más datos faltantes.
    - Priorizar problemas de calidad de datos.
    """
    return (
        df.isna().mean().mul(100).round(2)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "columna"})
        .sort_values("missing_pct", ascending=False)
    )


def duplicate_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resume la cantidad de filas duplicadas exactas del DataFrame.

    Qué hace:
    - Cuenta cuántas filas tiene el DataFrame.
    - Calcula cuántas filas están repetidas exactamente.
    - Calcula el porcentaje de duplicados exactos sobre el total.

    Para qué sirve:
    - Medir si el dataset tiene registros repetidos.
    - Detectar posibles problemas de carga, integración o captura.
    """
    rows_total = len(df)
    exact_duplicates = int(df.duplicated().sum())
    return pd.DataFrame({
        "filas_totales": [rows_total],
        "duplicados_exactos": [exact_duplicates],
        "pct_duplicados_exactos": [round(exact_duplicates / rows_total * 100, 2)]
    })


def cardinality_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen de cardinalidad por columna.

    Qué hace:
    - Recorre cada columna del DataFrame.
    - Obtiene su tipo de dato.
    - Cuenta cuántos nulos tiene.
    - Cuenta cuántos valores únicos contiene.
    - Calcula qué porcentaje representan esos valores únicos respecto al total de filas.

    Para qué sirve:
    - Identificar columnas casi únicas, como IDs.
    - Detectar columnas categóricas con pocas clases.
    - Encontrar columnas con demasiada variabilidad o posibles inconsistencias.
    """
    rows = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        rows.append({
            "columna": col,
            "dtype": str(df[col].dtype),
            "nulos": int(df[col].isna().sum()),
            "unicos": int(nunique),
            "pct_unicos_sobre_filas": round(nunique / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values(
        ["pct_unicos_sobre_filas", "columna"],
        ascending=[False, True]
    )


def iqr_outlier_summary(df: pd.DataFrame, numeric_cols=None) -> pd.DataFrame:
    """
    Detecta outliers en columnas numéricas usando la regla del IQR.

    Qué hace:
    - Si no se indican columnas, toma todas las numéricas.
    - Para cada columna calcula:
        * Q1 (percentil 25)
        * Q3 (percentil 75)
        * IQR = Q3 - Q1
        * límite inferior = Q1 - 1.5 * IQR
        * límite superior = Q3 + 1.5 * IQR
    - Cuenta cuántos valores quedan fuera de esos límites.
    - Calcula el porcentaje de outliers por columna.

    Para qué sirve:
    - Detectar valores extremos o atípicos.
    - Revisar posibles errores de captura o casos inusuales.
    - Priorizar revisión en variables numéricas.
    """
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()

    rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((s < lower) | (s > upper)).sum()

        rows.append({
            "columna": col,
            "q1": round(float(q1), 3),
            "q3": round(float(q3), 3),
            "iqr": round(float(iqr), 3),
            "limite_inferior": round(float(lower), 3),
            "limite_superior": round(float(upper), 3),
            "outliers_iqr": int(outliers),
            "pct_outliers_iqr": round(outliers / len(s) * 100, 2)
        })

    return pd.DataFrame(rows).sort_values("pct_outliers_iqr", ascending=False)


def top_values(df: pd.DataFrame, col: str, n: int = 10) -> pd.DataFrame:
    """
    Muestra los valores más frecuentes de una columna.

    Qué hace:
    - Cuenta cuántas veces aparece cada valor en la columna.
    - Incluye también los nulos.
    - Devuelve los n valores más frecuentes.

    Para qué sirve:
    - Revisar rápidamente la distribución de categorías.
    - Detectar valores dominantes, categorías raras o inconsistentes.
    - Entender mejor columnas de texto o categóricas.
    """
    return (
        df[col].value_counts(dropna=False).head(n)
        .rename("conteo").reset_index()
        .rename(columns={"index": col})
    )


def valid_email_simple(s: pd.Series) -> pd.Series:
    """
    Valida de forma básica si una serie tiene correos con estructura válida.

    Qué hace:
    - Reemplaza nulos por cadena vacía.
    - Evalúa cada valor con una expresión regular simple.
    - Devuelve una serie booleana:
        * True  -> parece un correo válido
        * False -> no cumple la estructura esperada

    Para qué sirve:
    - Detectar correos mal formados de manera rápida.
    - Aplicar una regla básica de calidad sobre columnas de email.

    Nota:
    - Es una validación simple de formato.
    - No garantiza que el correo exista realmente.
    """
    return s.fillna("").str.contains(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", regex=True)

In [0]:

# ============================================================
# 6. Métricas básicas de profiling
# ============================================================
from IPython.display import display, HTML

missing_df = percent_missing(raw)

dups_df = duplicate_summary(raw)

card_df = cardinality_summary(raw)

outliers_df = iqr_outlier_summary(raw)


display(HTML("<h3>Valores nulos</h3>"))
display(missing_df)
display(HTML("<br>"))

display(HTML("<h3>Duplicados</h3>"))
display(dups_df)
display(HTML("<br>"))

display(HTML("<h3>Cardinalidad</h3>"))
display(card_df.head(15))
display(HTML("<br>"))

display(HTML("<h3>Outliers</h3>"))
display(outliers_df)

In [0]:

categorical_cols = raw.select_dtypes(include=["object"]).columns.tolist()

print(categorical_cols)

for col in categorical_cols:
    print("\n" + "="*80)
    print(f"Top valores de: {col}")
    display(top_values(raw, col, n=10))

In [0]:

fecha_min = raw["fecha_venta"].min()
fecha_max = raw["fecha_venta"].max()

print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

fecha_corte = pd.Timestamp("2025-03-23")
fechas_futuras = (raw["fecha_venta"] > fecha_corte).sum()
print("Registros con fecha futura respecto a 2025-01-01:", fechas_futuras)

# Parte 4. Reglas de calidad del negocio

Además del profiling estadístico, conviene definir reglas de negocio.

In [0]:

# ============================================================
# 7. Reglas de calidad
# ============================================================

reglas = {
    "edad_valida_18_100": raw["edad_cliente"].between(18, 100, inclusive="both"),
    "ingreso_no_negativo": raw["ingreso_mensual"].fillna(0) >= 0,
    "precio_positivo": raw["precio_unitario"] > 0,
    "descuento_entre_0_y_1": raw["descuento_pct"].between(0, 1, inclusive="both"),
    "fecha_no_futura": raw["fecha_venta"] <= pd.Timestamp("2025-03-23"),
    "email_valido_simple": valid_email_simple(raw["email"])
}

quality_rows = []
for nombre, serie in reglas.items():
    invalidos = int((~serie).sum())
    quality_rows.append({
        "regla": nombre,
        "filas_invalidas": invalidos,
        "pct_invalidas": round(invalidos / len(raw) * 100, 2)
    })

quality_df = pd.DataFrame(quality_rows).sort_values("pct_invalidas", ascending=False)
quality_df.to_csv(DATA_QUALITY_CSV, index=False)
quality_df

In [0]:

plt.figure(figsize=(12, 5))
missing_df_sorted = missing_df.sort_values("missing_pct", ascending=False)
plt.bar(missing_df_sorted["columna"], missing_df_sorted["missing_pct"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("% de nulos")
plt.title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.show()

In [0]:

plt.figure(figsize=(12, 5))
plt.bar(outliers_df["columna"], outliers_df["pct_outliers_iqr"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("% outliers (IQR)")
plt.title("Porcentaje de outliers por columna")
plt.tight_layout()
plt.show()

# Parte 5. Profiling automático con ydata-profiling

La librería genera un reporte integral con:
- resumen general
- tipos de variables
- distribución de columnas
- nulos
- correlaciones
- advertencias
- duplicados
- ejemplos

In [0]:

# ============================================================
# 8. Reporte automático completo
# ============================================================

profile_raw = ProfileReport(
    raw,
    title="Reporte de Data Profiling - Dataset Crudo",
    explorative=True,
    minimal=False
)

profile_raw.to_file(PROFILE_HTML_RAW)
print("Reporte HTML generado en:", PROFILE_HTML_RAW)

In [0]:

# También puedes mostrar el reporte dentro de Colab
profile_raw

## Opción útil para datasets más pesados

Si el profiling completo se demora mucho, usa `minimal=True`.

In [0]:

# Perfil alternativo más liviano
profile_raw_minimal = ProfileReport(
    raw,
    title="Reporte de Data Profiling - Dataset Crudo (Minimal)",
    explorative=True,
    minimal=True
)

# Solo como ejemplo. Puedes guardar otro HTML si quieres.
print("Perfil minimal generado en memoria.")

# Parte 6. Documentación del dataset

No basta con perfilar: también hay que dejar el dataset explicado.

In [0]:

# ============================================================
# 9. Diccionario de datos
# ============================================================

descripciones = {
    "id_venta": "Identificador de la venta.",
    "fecha_venta": "Fecha en la que se registró la venta.",
    "ciudad": "Ciudad asociada al cliente o a la venta.",
    "canal": "Canal por el cual se realizó la venta.",
    "estado": "Estado operacional de la venta.",
    "segmento": "Segmento comercial del cliente.",
    "producto": "Producto adquirido.",
    "edad_cliente": "Edad reportada del cliente.",
    "ingreso_mensual": "Ingreso mensual estimado del cliente.",
    "cantidad": "Cantidad de productos en la venta.",
    "precio_unitario": "Precio unitario del producto.",
    "descuento_pct": "Porcentaje de descuento aplicado.",
    "email": "Correo electrónico del cliente.",
    "monto_bruto": "Valor bruto calculado antes del descuento.",
    "monto_neto": "Valor neto calculado después del descuento."
}

ejemplos = {}
for col in raw.columns:
    valores = raw[col].dropna().astype(str).head(3).tolist()
    ejemplos[col] = " | ".join(valores)

diccionario = pd.DataFrame({
    "columna": raw.columns,
    "descripcion": [descripciones.get(c, "") for c in raw.columns],
    "dtype_pandas": [str(raw[c].dtype) for c in raw.columns],
    "nulos": [int(raw[c].isna().sum()) for c in raw.columns],
    "pct_nulos": [round(raw[c].isna().mean() * 100, 2) for c in raw.columns],
    "unicos": [int(raw[c].nunique(dropna=True)) for c in raw.columns],
    "ejemplos": [ejemplos[c] for c in raw.columns]
})

diccionario.to_csv(DATA_DICTIONARY_CSV, index=False)
diccionario

In [0]:

# Exportar a Excel
xlsx_path = DOCS_DIR / "documentacion_dataset.xlsx"

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    diccionario.to_excel(writer, sheet_name="diccionario_datos", index=False)
    quality_df.to_excel(writer, sheet_name="reglas_calidad", index=False)
    missing_df.to_excel(writer, sheet_name="missing", index=False)
    outliers_df.to_excel(writer, sheet_name="outliers", index=False)

print("Excel generado en:", xlsx_path)

In [0]:

# ============================================================
# 10. Resumen ejecutivo del dataset
# ============================================================

summary_lines = []
summary_lines.append("# Resumen ejecutivo del dataset\n")
summary_lines.append(f"- Filas: **{len(raw):,}**\n")
summary_lines.append(f"- Columnas: **{raw.shape[1]}**\n")
summary_lines.append(f"- Duplicados exactos: **{int(raw.duplicated().sum()):,}**\n")
summary_lines.append("\n## Columnas con más nulos\n")

for _, row in missing_df.head(5).iterrows():
    summary_lines.append(f"- **{row['columna']}**: {row['missing_pct']}%\n")

summary_lines.append("\n## Reglas con más incumplimiento\n")
for _, row in quality_df.head(5).iterrows():
    summary_lines.append(f"- **{row['regla']}**: {row['pct_invalidas']}% inválidas\n")

summary_lines.append("\n## Observaciones clave\n")
summary_lines.append("- Hay problemas de consistencia en texto y categorías.\n")
summary_lines.append("- Existen fechas futuras, valores monetarios inválidos y correos mal formados.\n")
summary_lines.append("- Se recomienda una fase de estandarización antes de usar el dataset para analítica o modelado.\n")

DATASET_SUMMARY_MD.write_text("".join(summary_lines), encoding="utf-8")
print("Resumen Markdown generado en:", DATASET_SUMMARY_MD)
print(DATASET_SUMMARY_MD.read_text(encoding="utf-8"))

# Parte 7. Limpieza básica del dataset

No buscamos dejarlo perfecto.  
Buscamos mostrar cómo cambia el perfil después de una limpieza razonable.

In [0]:

# ============================================================
# 11. Limpieza
# ============================================================

clean = raw.copy()

for col in ["ciudad", "canal", "estado", "segmento", "producto"]:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

clean["ciudad"] = clean["ciudad"].replace({
    "bogotá": "bogota",
    " bogotá ": "bogota",
    "medellín": "medellin",
    "medellin": "medellin",
    "cali": "cali",
    "barranquilla": "barranquilla",
    "bucaramanga": "bucaramanga",
    "cartagena": "cartagena"
})

clean["canal"] = clean["canal"].replace({
    "app": "app",
    "call_center": "call_center",
    "tienda": "tienda",
    "web": "web"
})

clean["estado"] = clean["estado"].replace({
    "completada": "completada"
})

clean.loc[~clean["edad_cliente"].between(18, 100, inclusive="both"), "edad_cliente"] = np.nan
clean.loc[clean["ingreso_mensual"] < 0, "ingreso_mensual"] = np.nan
clean.loc[clean["precio_unitario"] <= 0, "precio_unitario"] = np.nan
clean.loc[~clean["descuento_pct"].between(0, 1, inclusive="both"), "descuento_pct"] = np.nan
clean.loc[clean["fecha_venta"] > pd.Timestamp("2025-01-01"), "fecha_venta"] = pd.NaT
clean.loc[~valid_email_simple(clean["email"]), "email"] = np.nan

clean = clean.drop_duplicates().copy()

clean["monto_bruto"] = (clean["cantidad"] * clean["precio_unitario"]).round(2)
clean["monto_neto"] = (clean["monto_bruto"] * (1 - clean["descuento_pct"])).round(2)

for col in ["ciudad", "canal", "estado", "segmento", "producto"]:
    clean[col] = clean[col].astype("category")

clean["cantidad"] = pd.to_numeric(clean["cantidad"], downcast="integer")
clean["edad_cliente"] = pd.to_numeric(clean["edad_cliente"], downcast="float")
clean["ingreso_mensual"] = pd.to_numeric(clean["ingreso_mensual"], downcast="float")
clean["precio_unitario"] = pd.to_numeric(clean["precio_unitario"], downcast="float")
clean["descuento_pct"] = pd.to_numeric(clean["descuento_pct"], downcast="float")
clean["monto_bruto"] = pd.to_numeric(clean["monto_bruto"], downcast="float")
clean["monto_neto"] = pd.to_numeric(clean["monto_neto"], downcast="float")

clean.to_csv(CLEAN_CSV, index=False)
print("Dataset limpio guardado en:", CLEAN_CSV)
clean.head()

In [0]:

def df_mem_mb(df):
    return round(df.memory_usage(deep=True).sum() / 1024**2, 2)

comparacion = pd.DataFrame([
    {
        "dataset": "raw",
        "filas": len(raw),
        "columnas": raw.shape[1],
        "memoria_mb": df_mem_mb(raw),
        "duplicados_exactos": int(raw.duplicated().sum()),
        "nulos_totales": int(raw.isna().sum().sum())
    },
    {
        "dataset": "clean",
        "filas": len(clean),
        "columnas": clean.shape[1],
        "memoria_mb": df_mem_mb(clean),
        "duplicados_exactos": int(clean.duplicated().sum()),
        "nulos_totales": int(clean.isna().sum().sum())
    }
])

comparacion["reduccion_memoria_pct"] = (
    (comparacion.loc[0, "memoria_mb"] - comparacion["memoria_mb"]) / comparacion.loc[0, "memoria_mb"] * 100
).round(2)

comparacion.to_csv(COMPARISON_CSV, index=False)
comparacion

In [0]:

# ============================================================
# 12. Profiling del dataset limpio
# ============================================================

profile_clean = ProfileReport(
    clean,
    title="Reporte de Data Profiling - Dataset Limpio",
    explorative=True,
    minimal=False
)

profile_clean.to_file(PROFILE_HTML_CLEAN)
print("Reporte HTML generado en:", PROFILE_HTML_CLEAN)

In [0]:

quality_clean_rows = []
reglas_clean = {
    "edad_valida_18_100": clean["edad_cliente"].between(18, 100, inclusive="both") | clean["edad_cliente"].isna(),
    "ingreso_no_negativo": clean["ingreso_mensual"].ge(0) | clean["ingreso_mensual"].isna(),
    "precio_positivo": clean["precio_unitario"].gt(0) | clean["precio_unitario"].isna(),
    "descuento_entre_0_y_1": clean["descuento_pct"].between(0, 1, inclusive="both") | clean["descuento_pct"].isna(),
    "fecha_no_futura": clean["fecha_venta"].le(pd.Timestamp("2025-01-01")) | clean["fecha_venta"].isna(),
    "email_valido_simple": valid_email_simple(clean["email"]) | clean["email"].isna()
}

for nombre, serie in reglas_clean.items():
    invalidos = int((~serie).sum())
    quality_clean_rows.append({
        "regla": nombre,
        "filas_invalidas": invalidos,
        "pct_invalidas": round(invalidos / len(clean) * 100, 2)
    })

quality_clean_df = pd.DataFrame(quality_clean_rows).sort_values("pct_invalidas", ascending=False)

comparison_quality = quality_df.merge(
    quality_clean_df,
    on="regla",
    suffixes=("_raw", "_clean")
)
comparison_quality["mejora_pct_points"] = (
    comparison_quality["pct_invalidas_raw"] - comparison_quality["pct_invalidas_clean"]
).round(2)

display(comparacion)
display(comparison_quality)

In [0]:

plt.figure(figsize=(7, 4))
plt.bar(comparacion["dataset"], comparacion["memoria_mb"])
plt.ylabel("Memoria (MB)")
plt.title("Memoria del DataFrame: crudo vs limpio")
plt.tight_layout()
plt.show()

In [0]:

plt.figure(figsize=(7, 4))
plt.bar(comparacion["dataset"], comparacion["nulos_totales"])
plt.ylabel("Nulos totales")
plt.title("Nulos totales: crudo vs limpio")
plt.tight_layout()
plt.show()

# Parte 8. Descargar los entregables desde Colab

Si no usaste Drive, puedes descargar archivos directamente a tu computador.

# Conclusiones didácticas

- **Pandas** te ayuda a construir un profiling manual y entendible.
- **ydata-profiling** acelera muchísimo la exploración.
- Ninguna herramienta automática reemplaza el criterio del analista.
- El profiling no es solo mirar estadísticas: también es validar reglas de negocio.
- La documentación del dataset es parte del trabajo profesional, no un extra.

> Si un dataset solo lo entiende quien lo creó, entonces todavía no está bien documentado.